# Simulation Metrics

Compute ADE/FDE, offroad, collision proxy, and kinematic violations on toy trajectories.

In [ ]:
import torch
import matplotlib.pyplot as plt
torch.manual_seed(0)

In [ ]:
def ade(pred, target):
    return torch.norm(pred - target, dim=-1).mean()

def fde(pred, target):
    return torch.norm(pred[:, -1] - target[:, -1], dim=-1).mean()

def offroad(traj, x_min=0, x_max=10, y_min=-2, y_max=6):
    x, y = traj[..., 0], traj[..., 1]
    return ((x < x_min) | (x > x_max) | (y < y_min) | (y > y_max)).any(dim=1)

def kinematic_flags(traj, dt=0.1, max_speed=40.0, max_accel=8.0):
    vel = (traj[:, 1:] - traj[:, :-1]) / dt
    speed = torch.norm(vel, dim=-1)
    accel = (speed[:, 1:] - speed[:, :-1]) / dt
    return (speed > max_speed).any(dim=1), (accel.abs() > max_accel).any(dim=1)

def collision_proxy(a, b, radius=1.5):
    # Point-distance proxy. Real systems use oriented boxes.
    return (torch.norm(a - b, dim=-1) < radius).any(dim=1)

In [ ]:
T = 20
t = torch.linspace(0, 1, T)
target = torch.stack([5 * t, torch.zeros_like(t)], dim=-1)[None]
pred_good = target + 0.1 * torch.randn_like(target)
pred_offroad = torch.stack([5 * t, 8 * t], dim=-1)[None]
pred_jump = target.clone(); pred_jump[:, 10:] += torch.tensor([8.0, 0.0])

preds = torch.cat([pred_good, pred_offroad, pred_jump], dim=0)
names = ['good', 'offroad', 'jump']
target3 = target.expand_as(preds)

print('ADE:', ade(preds, target3).item())
print('FDE:', fde(preds, target3).item())
print('offroad:', dict(zip(names, offroad(preds).tolist())))
speed_bad, accel_bad = kinematic_flags(preds)
print('speed_bad:', dict(zip(names, speed_bad.tolist())))
print('accel_bad:', dict(zip(names, accel_bad.tolist())))

plt.figure(figsize=(5, 4))
for p, n in zip(preds, names):
    plt.plot(p[:, 0], p[:, 1], marker='o', label=n)
plt.legend(); plt.axis('equal'); plt.title('Toy metric cases'); plt.show()